Importation de donnees

In [10]:
import pandas as pd
df = pd.read_csv('football.csv')
df_comp=pd.read_csv("competition.csv")

In [11]:
df

,id,date,home_team,away_team,rank_change_home,rank_change_away,home_goals_mean,home_goals_mean_l5,home_goals_suf_mean,home_goals_suf_mean_l5,...,away_goals_mean,away_goals_mean_l5,away_goals_suf_mean,away_goals_suf_mean_l5,away_rank_mean,away_rank_mean_l5,away_points_mean,away_points_mean_l5,target,match_type
0,2221,2021-09-09,Brazil,Peru,-1.0,-5.0,2.028571,0.8,0.371429,0.4,...,1.027778,1.4,1.583333,1.6,24.388889,21.2,8.16,31.28,0,Friendly
1,507,2019-05-29,Comoros,Mauritius,5.0,NaN,1.166667,1.2,1.666667,1.8,...,1.200000,1.2,1.200000,1.2,159.800000,159.8,-6.00,-6.00,0,Friendly
2,1545,2020-11-18,Serbia,Russia,-1.0,NaN,1.608696,1.2,1.347826,1.2,...,2.136364,0.8,1.045455,1.2,65.681818,62.0,70.00,-5.00,0,Friendly
3,2067,2021-09-01,Senegal,Togo,-1.0,NaN,1.482759,1.4,0.620690,0.8,...,0.857143,0.8,1.214286,1.2,104.500000,82.0,-31.86,-22.86,0,Friendly
4,4448,2024-03-22,Peru,Nicaragua,-2.0,NaN,0.968750,0.2,1.265625,1.6,...,1.600000,3.0,1.244444,0.2,132.200000,169.4,-21.30,23.59,0,Competition
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,2020,2021-07-12,Jamaica,Suriname,0.0,NaN,2.000000,1.0,1.090909,2.0,...,3.181818,3.4,NaN,1.0,154.636364,153.6,53.43,37.43,0,Friendly
3773,696,2019-06-28,Colombia,Chile,1.0,NaN,1.909091,2.0,0.545455,0.0,...,1.545455,1.8,1.272727,0.8,39.090909,44.2,-9.00,-4.00,1,Friendly
3774,2325,2021-10-11,Mozambique,Cameroon,3.0,4.0,1.000000,0.6,1.153846,1.6,...,1.086957,1.2,0.478261,0.2,88.260870,86.2,-11.54,-22.54,1,Friendly
3775,3301,2022-11-19,Colombia,Paraguay,0.0,NaN,1.319149,2.4,0.872340,0.6,...,0.909091,0.8,1.272727,0.6,34.886364,31.2,-28.23,5.47,0,Competition


Entrainement du modele.

In [12]:
df = df.drop(columns=['date','id','home_team','away_team','match_type'])

Definir le X et y

In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
x=df.drop(columns=['target'])
y=df['target']
x_train, x_test, y_train, y_test= train_test_split(x,y, test_size=0.2,random_state=2)



In [15]:
from sklearn.tree import DecisionTreeClassifier
dtc = DecisionTreeClassifier(max_depth=3,min_samples_leaf=5)
dtc.fit(x_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [16]:
from sklearn.metrics import f1_score
f1_score(y_test, dtc.predict(x_test))

0.6240601503759399

voir l'importance des colonnes

In [17]:
dtc.feature_importances_


array([0.        , 0.        , 0.1712291 , 0.        , 0.29361562,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.07402828, 0.10371346, 0.35741354, 0.        , 0.        ,
       0.        , 0.        , 0.        ])

In [18]:
import numpy as np
x.columns[np.argsort(dtc.feature_importances_)[-3:][::-1]]

Index(['away_goals_suf_mean', 'home_goals_suf_mean', 'home_goals_mean'], dtype='object')

Prediction sur les 3 meilleurs colonnes

In [20]:
top3_cols = x.columns[np.argsort(dtc.feature_importances_)[-3:][::-1]]
x_reduit_train = x_train[top3_cols]
x_reduit_test = x_test[top3_cols]

dtc2 = DecisionTreeClassifier(
    max_depth=5,           
    min_samples_split=20,  
    min_samples_leaf=15,    
    max_features=5         
)
dtc2.fit(x_reduit_train, y_train)



,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,20
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,5
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [21]:
from sklearn.metrics import f1_score
f1_score(y_test, dtc2.predict(x_reduit_test))

0.6807563959955506

Predicition sur competition.csv

In [22]:
df_comp = df_comp.drop(columns=['date','home_team','away_team','match_type'])

In [23]:
df_comp_reduit = df_comp[top3_cols] 

predictions = dtc2.predict(df_comp_reduit)


In [25]:

soumission = pd.DataFrame({'id': df_comp['id'],'prediction': predictions})

soumission.to_csv('DecisionTree.csv', index=False)


Entrainement de Forest Aleatoire

In [26]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(x_train,y_train)
f1_score(y_test, rf.predict(x_test))

0.6793893129770993

In [27]:
rf.feature_importances_

array([0.02761812, 0.024179  , 0.0766211 , 0.03863412, 0.07523039,
       0.04723716, 0.06836169, 0.05652234, 0.05909441, 0.04848323,
       0.07158388, 0.04581065, 0.08590716, 0.04453243, 0.07458057,
       0.05536499, 0.0537391 , 0.04649965])

In [28]:
x.columns[np.argsort(rf.feature_importances_)[-3:][::-1]] 

Index(['away_goals_suf_mean', 'home_goals_mean', 'home_goals_suf_mean'], dtype='object')

In [29]:


resultat = []
for sousmodel in rf.estimators_:
    resultat.append(f1_score(y_test, sousmodel.predict(x_test)))
    

c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\segui\anaconda3\Lib\site-packages\sklearn\utils\val

0.6074649271994275


In [30]:
print(np.mean(resultat))

0.6074649271994275


In [52]:
df_comp_features = df_comp[x_train.columns].copy()

rf_final = RandomForestClassifier(bootstrap=False, criterion='gini', max_depth=9, 
                                  max_features='sqrt', max_leaf_nodes=50,
                                  min_samples_leaf=2, min_samples_split=5, 
                                  n_estimators=200, random_state=22)


rf_final.fit(x_train, y_train)
score = f1_score(y_test, rf_final.predict(x_test))
print(score)



0.6945812807881774


In [56]:
df_comp_features = df_comp[x_train.columns].copy()
predictions2 = rf_final.predict(df_comp_features)

In [59]:

soumission = pd.DataFrame({'id': df_comp['id'],'target': predictions2})

soumission.to_csv('RandomForest.csv', index=False)